# XPOS Data Augmentation — Gemini-based
Scans `tor_train.tsv` for rare XPOS tags, collects few-shot examples,
and asks Gemini to generate new Torlak-dialect sentences containing each rare tag.

**Output:** `tor_augmented_train.tsv` and `tor_augmented_dev.tsv` in the same `dataset/` folder.

In [ ]:
!pip install google-generativeai -q

In [ ]:
import os, re, time, random
from collections import defaultdict, Counter
import google.generativeai as genai
from google.colab import drive, userdata

drive.mount('/content/drive')

# ── CONFIG ─────────────────────────────────────────────────────────────────────
DRIVE_ROOT      = '/content/drive/MyDrive/TorlakTag'
TRAIN_TSV       = f'{DRIVE_ROOT}/dataset/tor_train.tsv'
AUG_TRAIN_TSV   = f'{DRIVE_ROOT}/dataset/tor_augmented_train.tsv'
AUG_DEV_TSV     = f'{DRIVE_ROOT}/dataset/tor_augmented_dev.tsv'

GEMINI_MODEL      = 'gemini-2.5-flash'
RARE_THRESHOLD    = 25
SENTENCES_PER_TAG = 12   # doubled — more attempts for tags Gemini struggles with
FEW_SHOT_SENTS    = 3
DEV_RATIO         = 0.15
SLEEP_BETWEEN     = 2.0
APPEND_MODE       = True
SEED              = 42

# Threshold logic:
#   orig=0 tags : need at least 3 (any example beats none)
#   orig>0 tags : need at least 5
ADEQUATE_THRESHOLD_NEW      = 3
ADEQUATE_THRESHOLD_EXISTING = 5

random.seed(SEED)

GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
genai.configure(api_key=GEMINI_API_KEY)
client = genai.GenerativeModel(GEMINI_MODEL)
print('Gemini client ready')

In [ ]:
def load_tsv(path):
    """Parse form\tlemma\txpos TSV; blank/tab-only lines = sentence boundaries."""
    sentences, current = [], []
    with open(path, encoding='utf-8') as f:
        for line in f:
            line = line.rstrip('\n')
            if not line.strip():
                if current:
                    sentences.append(current)
                    current = []
            else:
                parts = line.split('\t')
                if len(parts) == 3 and parts[0].strip():
                    current.append({'form': parts[0], 'lemma': parts[1], 'xpos': parts[2]})
    if current:
        sentences.append(current)
    return sentences

train_sents = load_tsv(TRAIN_TSV)
total_toks  = sum(len(s) for s in train_sents)
print(f'Loaded {len(train_sents)} training sentences, {total_toks} tokens')

In [ ]:
# ── Count tag frequencies ───────────────────────────────────────────────────────
tag_counts = Counter(tok['xpos'] for sent in train_sents for tok in sent)

# ── Build index: tag → list of sentences that contain it ───────────────────────
tag_to_sents = defaultdict(list)
for sent in train_sents:
    seen = set()
    for tok in sent:
        if tok['xpos'] not in seen:
            tag_to_sents[tok['xpos']].append(sent)
            seen.add(tok['xpos'])

# ── Load current augmented files to check what's already been covered ──────────
existing_aug = []
for p in [AUG_TRAIN_TSV, AUG_DEV_TSV]:
    if os.path.exists(p):
        existing_aug += load_tsv(p)
aug_so_far = Counter(tok['xpos'] for s in existing_aug for tok in s)

# ── FORCE_INCLUDE — full list of target tags ────────────────────────────────────
FORCE_INCLUDE = {
    # Round 1 — previously 0%-accuracy tags
    'Pp3msg', 'Pd-fsn', 'Pp1-pa', 'Npmsay', 'Npmsg',
    'Agpfpny', 'Agpfsn', 'Agpmpn', 'Pi3m-n', 'Agpfpay',
    'Ncfsn-t', 'Ncfpa-t', 'Vme3p',
    'Mlonsn', 'Agpnsny', 'Ps1mpn', 'Ncmpg', 'Npmpn',
    'Vcp-sn', 'Ncfpn', 'Ncnsn', 'Agpmsny', 'Npmsn',
    # Round 2 — UNK tags from test set
    'Ncmpl', 'Vme3s-y', 'Appfsnn', 'Ncfpay', 'Ps1msg',
    'Ncmpgn', 'Agpmpay-t', 'Ncmsnnt', 'Ncnsay', 'Pp1mpa',
    'Ncfsant', 'Pq-n-n', 'Vmcp-sf', 'Ps1nsn-t', 'Vmc-sn',
    'Agcfsan', 'Va', 'Pp1-say', 'Pq-fsa', 'Ncmpgy',
    'Ncmsvn', 'Ncmsln', 'Vmf1p', 'Ncfsnnt', 'Ncfslnt',
    'Ncfsnt', 'Ncmslnt', 'Ncfsgnt', 'Ncnsgyt', 'Ncnsny',
    'Agpfsayn', 'Ncnpnnt', 'Ncnpa-t', 'Agsnsny', 'Ncmsnyt',
    'Ncmsayt', 'Ncfsg-t', 'Mlcfsn-t', 'Ncnsant', 'Ncmsyv',
    'Npmsnn-t', 'Mdc', 'Vae3p',
}

EXCLUDE = {'SI', 'Apcfsnny', 'POS_CORR', '[UNK]'}

# ── Select only tags that are still below threshold (second-pass mode) ──────────
def is_adequate(tag):
    orig  = tag_counts.get(tag, 0)
    added = aug_so_far.get(tag, 0)
    total = orig + added
    threshold = ADEQUATE_THRESHOLD_NEW if orig == 0 else ADEQUATE_THRESHOLD_EXISTING
    return total >= threshold

all_candidates = sorted(
    {t for t, c in tag_counts.items() if c < RARE_THRESHOLD} | FORCE_INCLUDE
)
all_candidates = [t for t in all_candidates if t and t[0] not in ('Z', 'X') and t not in EXCLUDE]

# Only target tags that still need more data
rare_tags = [t for t in all_candidates if not is_adequate(t)]

print(f'Total candidates   : {len(all_candidates)}')
print(f'Already adequate   : {len(all_candidates) - len(rare_tags)}')
print(f'Still needs more   : {len(rare_tags)}')
print()
for t in rare_tags:
    orig  = tag_counts.get(t, 0)
    added = aug_so_far.get(t, 0)
    thresh = ADEQUATE_THRESHOLD_NEW if orig == 0 else ADEQUATE_THRESHOLD_EXISTING
    print(f'  {t:<24}  orig={orig:3d}  aug={added:3d}  total={orig+added:3d}  need={thresh}')

In [ ]:
def sent_to_tsv(sent):
    """Convert a sentence (list of dicts) back to TSV block."""
    return '\n'.join(f"{t['form']}\t{t['lemma']}\t{t['xpos']}" for t in sent)

def pick_few_shots(tag, n=FEW_SHOT_SENTS):
    """Return up to n example TSV blocks from training data."""
    pool = tag_to_sents.get(tag, [])
    chosen = random.sample(pool, min(n, len(pool)))
    # Prefer shorter sentences so the prompt stays concise
    chosen.sort(key=len)
    return [sent_to_tsv(s) for s in chosen]

# Quick sanity
ex = pick_few_shots('Ncfpn')
print(f'Example few-shots for Ncfpn ({len(ex)} sentences):')
for block in ex:
    print(block)
    print()

In [ ]:
SYSTEM_PROMPT = """You are an expert annotator of Torlak dialect Serbian with the MTE (Multext-East) morphological tagset.

Rules:
- Write natural-sounding Torlak speech (use dialect forms, vowel reductions, and local lexicon — NOT standard Serbian).
- Output ONLY tab-separated TSV: form<TAB>lemma<TAB>xpos, one token per line.
- Separate sentences with a single blank line.
- Annotate EVERY token in the sentence — do not skip punctuation.
- Each sentence MUST contain at least one token tagged with the target XPOS.
- Do not add any commentary, headers, or markdown — raw TSV only."""

def build_prompt(tag, few_shots, n_sentences=SENTENCES_PER_TAG):
    fs_block = '\n\n'.join(few_shots) if few_shots else '(no examples available)'
    return (
        f"Target XPOS tag: {tag}\n"
        f"Generate {n_sentences} Torlak dialect sentences, each containing "
        f"at least one token tagged '{tag}'.\n\n"
        f"Reference sentences from the corpus that contain this tag:\n"
        f"---\n{fs_block}\n---\n\n"
        f"Output {n_sentences} new annotated sentences:"
    )

def call_gemini(prompt, retries=3):
    for attempt in range(retries):
        try:
            resp = client.generate_content(
                [SYSTEM_PROMPT, prompt],
                generation_config=genai.types.GenerationConfig(
                    temperature=0.8,
                    max_output_tokens=1024,
                )
            )
            return resp.text
        except Exception as e:
            print(f'  Attempt {attempt+1} failed: {e}')
            time.sleep(5 * (attempt + 1))
    return ''

print('Prompt example:')
print(build_prompt('Ncfpn', pick_few_shots('Ncfpn')))

In [ ]:
# Known XPOS prefixes — used to catch obviously bad tags
VALID_FIRST_CHARS = set('NVAPMRSCQIZXMl')

def parse_tsv_response(text, target_tag):
    """
    Parse Gemini's raw TSV output into a list of sentences.
    Each sentence is a list of dicts {form, lemma, xpos}.
    Sentences where the target tag is absent are discarded.
    Tokens with malformed tags are skipped (not the whole sentence).
    """
    sentences, current = [], []
    for raw_line in text.splitlines():
        line = raw_line.strip()
        if not line:
            if current:
                sentences.append(current)
                current = []
            continue
        # Strip markdown fences if Gemini wraps output
        if line.startswith('```'):
            continue
        parts = line.split('\t')
        if len(parts) != 3:
            continue
        form, lemma, xpos = parts
        form, lemma, xpos = form.strip(), lemma.strip(), xpos.strip()
        if not form or not xpos:
            continue
        # Basic sanity: xpos must start with a letter, no spaces
        if ' ' in xpos or not xpos[0].isalpha():
            continue
        current.append({'form': form, 'lemma': lemma, 'xpos': xpos})
    if current:
        sentences.append(current)

    # Keep only sentences that contain the target tag
    valid = [s for s in sentences if any(t['xpos'] == target_tag for t in s)]
    return valid

# ── Run augmentation ────────────────────────────────────────────────────────────
all_augmented = []   # flat list of sentence dicts
tag_results   = {}   # tag → generated sentences (for inspection)

for idx, tag in enumerate(rare_tags):
    print(f'[{idx+1}/{len(rare_tags)}] {tag} (train_count={tag_counts.get(tag, 0)}) …', end=' ')
    few_shots = pick_few_shots(tag)
    prompt    = build_prompt(tag, few_shots)
    raw       = call_gemini(prompt)
    parsed    = parse_tsv_response(raw, tag)
    print(f'{len(parsed)} valid sentences')

    tag_results[tag] = parsed
    all_augmented.extend(parsed)
    time.sleep(SLEEP_BETWEEN)

print(f'\nTotal augmented sentences: {len(all_augmented)}')
print(f'Total augmented tokens   : {sum(len(s) for s in all_augmented)}')

In [ ]:
# ── Spot-check: show a few generated sentences per tag ─────────────────────────
for tag in list(rare_tags)[:6]:
    sents = tag_results.get(tag, [])
    print(f'=== {tag} ({len(sents)} sentences) ===')
    for s in sents[:2]:
        for tok in s:
            mark = ' ◄' if tok['xpos'] == tag else ''
            print(f"  {tok['form']:<20} {tok['lemma']:<20} {tok['xpos']}{mark}")
        print()

In [ ]:
import math, os

# ── Shuffle and split into augmented train / dev ────────────────────────────────
random.shuffle(all_augmented)
n_dev     = max(1, math.ceil(len(all_augmented) * DEV_RATIO))
aug_dev   = all_augmented[:n_dev]
aug_train = all_augmented[n_dev:]

def write_tsv(path, sentences, append=False):
    mode = 'a' if append and os.path.exists(path) else 'w'
    with open(path, mode, encoding='utf-8') as f:
        for sent in sentences:
            for tok in sent:
                f.write(f"{tok['form']}\t{tok['lemma']}\t{tok['xpos']}\n")
            f.write('\n')

write_tsv(AUG_TRAIN_TSV, aug_train, append=APPEND_MODE)
write_tsv(AUG_DEV_TSV,   aug_dev,   append=APPEND_MODE)

action = 'Appended to' if APPEND_MODE else 'Saved'
print(f'{action} augmented train: {len(aug_train)} new sentences → {AUG_TRAIN_TSV}')
print(f'{action} augmented dev  : {len(aug_dev)} new sentences → {AUG_DEV_TSV}')

# ── Full coverage verification (reads final merged files) ──────────────────────
final_aug = load_tsv(AUG_TRAIN_TSV) + load_tsv(AUG_DEV_TSV)
aug_total  = Counter(tok['xpos'] for s in final_aug for tok in s)

print(f'\n{"Tag":<24}  {"orig":>5}  {"aug":>5}  {"total":>6}  {"need":>5}  status')
print('-' * 70)

all_ok, needs_more = [], []
for tag in sorted(FORCE_INCLUDE):
    orig   = tag_counts.get(tag, 0)
    added  = aug_total.get(tag, 0)
    total  = orig + added
    thresh = ADEQUATE_THRESHOLD_NEW if orig == 0 else ADEQUATE_THRESHOLD_EXISTING
    if total >= thresh:
        status = 'OK'
        all_ok.append(tag)
    else:
        status = f'NEEDS MORE'
        needs_more.append(tag)
    print(f'{tag:<24}  {orig:>5}  {added:>5}  {total:>6}  {thresh:>5}  {status}')

print(f'\nAdequately covered : {len(all_ok)}/{len(FORCE_INCLUDE)}')
if needs_more:
    print(f'Still needs more   : {len(needs_more)} — re-run gemini-augment + save-files cells')
    print('  ' + ', '.join(sorted(needs_more)))
else:
    print('All target tags adequately covered — ready to retrain!')